# E3 — CatBoost

[Artículo](https://proceedings.neurips.cc/paper/2018/hash/14491b756b3a51daac41c24863285549-Abstract.html). Experimento solicitado de la búsqueda Scopus.

Se usa `CatBoostRegressor` oficial, pérdida RMSE y
`boosting_type='Ordered'`, categorías nativas y semilla42. Se imputan medianas
numéricas y modas categóricas dentro de cada fit, sin one-hot, escalado o
logaritmos externos. El identificador de estudiante nunca es predictor.

Grilla24: iteraciones300/600 × profundidad2/4/6 × learning_rate.03/.1 ×
l2_leaf_reg3/10. No hay early stopping con test ni eval_set externo.
Un adaptador sklearn evita un problema de clone(cat_features) con sklearn1.9;
el entrenamiento interno sigue siendo íntegramente CatBoost.

## Protocolo compartido

Misma fuente real `Data/Datos_Brutos.csv`, reglas de negocio y15 predictores.
Objetivo `nota_primer_parcial`; nota_final, pidm, periodo e índice de fila no
son entradas. Sin aumentación ni eliminación de outliers válidos.
Train1309 filas/1293 estudiantes; test202530:188 estudiantes, sin solapamiento.

Cinco folds externos canónicos por estudiante, cinco internos en cada búsqueda
y cinco internos para elegir parámetros finales. RMSE medio interno selecciona
la configuración. Cada preprocesador y modelo se ajusta solo en su fit.
La selección final se guarda antes de predecir test. Mismas funciones del tablero:
RMSE, MAE, MAPE(%) y R²; métricas sin ponderación, escala original.
Media/SD entre folds, OOF reunido y test temporal se informan por separado.

Se guardan grillas, auditorías, hashes, OOF y predicciones test. El código puede
reanudar cálculos de igual procedencia; si cambia la procedencia exige una
carpeta nueva. Pesos/estimadores quedan locales en modelos_preentrenados/scopus.
No se reemplazan los modelos del dashboard ni se reescriben informes anteriores.

In [1]:
from pathlib import Path
import sys
BOOK=Path.cwd() if (Path.cwd()/'entregable3.py').exists() else Path.cwd()/'Precalculo'
sys.path.insert(0,str(BOOK.resolve()))
import pandas as pd
from IPython.display import display
from evaluar_scopus_e3 import run,compare,ROOT

report=run('catboost')
display(pd.DataFrame(report['cv']).round(6))
print('Parámetros finales:',report['best_params'])

,metrica,media_cv,sd_cv,OOF_agrupado
0,RMSE,0.816276,0.036602,0.817092
1,MAE,0.646609,0.035132,0.646611
2,MAPE (%),24.311994,2.178608,24.311495
3,R²,0.463267,0.051582,0.465476


Parámetros finales: {'depth': 6, 'iterations': 300, 'l2_leaf_reg': 3.0, 'learning_rate': 0.03}


## Test temporal y auditoría

El test no selecciona hiperparámetros.

In [2]:
display(pd.DataFrame([report['test']]).round(6))
display(pd.DataFrame(report['fold_audit']))

,RMSE,MAE,MAPE (%),R²
0,0.905669,0.731892,28.722883,0.455368


,fold,best_params,inner_rmse,fit_rows,validation_rows,shared_students,seconds
0,1,"{'depth': 6, 'iterations': 600, 'l2_leaf_reg':...",0.836106,1047,262,0,775.601911
1,2,"{'depth': 4, 'iterations': 600, 'l2_leaf_reg':...",0.808115,1047,262,0,1232.039009
2,3,"{'depth': 6, 'iterations': 300, 'l2_leaf_reg':...",0.806547,1047,262,0,1235.878520
3,4,"{'depth': 6, 'iterations': 300, 'l2_leaf_reg':...",0.815895,1047,262,0,1226.966478
4,5,"{'depth': 2, 'iterations': 600, 'l2_leaf_reg':...",0.815495,1048,261,0,1217.774475


## Comparación conjunta

Tras completar los cuatro modelos, consultar precalculo_scopus_e3.ipynb y resultados/e3/scopus/comparacion.json.